# 06 · Layer 2：App、Callbacks 與 Plugins

前面的東西都在講「agent 做什麼」。這一章講**橫切關注**——那些每個 agent 都
需要、但不屬於任何單一 agent 的事：日誌、稽核、成本控制、安全護欄、重試。

ADK 給了兩層機制：

| | 掛在哪 | 範圍 |
|---|---|---|
| **Callback** | 單一 agent 或 tool | 只有那個物件 |
| **Plugin** | App / Runner | **全域，所有 agent** |

而它們之間有一條**很容易記反、記反就會 debug 到懷疑人生**的規則。
本章會用程式碼把它證明出來。

## 0. 環境

In [1]:
import warnings

warnings.filterwarnings("ignore")

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from shared import ask, get_model, new_session, peek_state, print_state, quiet, run_once

quiet()

## 1. `App`：ADK 2.x 的應用層容器

前幾章我們都是 `Runner(agent=..., session_service=...)`。ADK 2.x 多了一層
`App`，把「這個應用的設定」跟「怎麼執行」分開。

有些功能**只能設在 `App` 上，設在 agent 上沒有用**：

In [2]:
from google.adk.apps import App

print("App 可設定的欄位:")
for field in App.model_fields:
    print(f"  {field}")

App 可設定的欄位:
  name
  root_agent
  plugins
  events_compaction_config
  context_cache_config
  resumability_config


| 欄位 | 用途 | 對應的 30 天 |
|---|---|---|
| `plugins` | 全域的橫切邏輯 | Day 12 |
| `events_compaction_config` | 上下文壓縮（省 token） | Day 10 |
| `context_cache_config` | 內容快取（省錢） | Day 11 |
| `resumability_config` | 可暫停／可恢復的執行 | Day 24 |

In [3]:
from google.adk.agents import LlmAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService

APP = "concept_track"


def get_order_status(order_id: str) -> dict:
    """查詢訂單狀態。

    Args:
        order_id: 訂單編號。
    """
    return {"order_id": order_id, "status": "已出貨"}


agent = LlmAgent(
    name="support",
    model=get_model(),
    instruction="你是客服，查訂單一定要用工具。用繁體中文回答。",
    tools=[get_order_status],
)

app = App(name=APP, root_agent=agent)
runner = Runner(app=app, session_service=InMemorySessionService())

sid = await new_session(runner)
print(await ask(runner, "查一下訂單 A-1001", session_id=sid))

您的訂單 A-1001 目前的狀態是：「已出貨」。


## 2. Callback：六個（其實是八個）掛載點

`LlmAgent` 身上可以掛這些鉤子：

In [4]:
print("LlmAgent 的 callback 欄位:")
for field in LlmAgent.model_fields:
    if "callback" in field:
        print(f"  {field}")

LlmAgent 的 callback 欄位:
  before_agent_callback
  after_agent_callback
  before_model_callback
  after_model_callback
  on_model_error_callback
  before_tool_callback
  after_tool_callback
  on_tool_error_callback


執行順序是一個巢狀結構：

```
before_agent_callback
  └─ before_model_callback     ← 可以改 request，或直接回傳假答案跳過模型
       (呼叫模型)
     after_model_callback      ← 可以改回應
     └─ before_tool_callback   ← 可以改參數，或直接回傳假結果跳過工具
          (執行工具)
        after_tool_callback    ← 可以改結果
after_agent_callback
```

**回傳值決定行為**：回 `None`＝放行；回一個物件＝**短路**，用你的回傳值取代
真正的呼叫。這就是護欄的實作方式。

In [5]:
from google.adk.tools import BaseTool, ToolContext

AUDIT = []


def audit_before_tool(tool: BaseTool, args: dict, tool_context: ToolContext):
    AUDIT.append(f"呼叫 {tool.name} args={args}")
    return None  # 放行


def audit_after_tool(tool: BaseTool, args: dict, tool_context: ToolContext, tool_response: dict):
    AUDIT.append(f"{tool.name} 回傳 {tool_response}")
    return None


audited = LlmAgent(
    name="audited_support",
    model=get_model(),
    instruction="你是客服，查訂單一定要用工具。用繁體中文回答。",
    tools=[get_order_status],
    before_tool_callback=audit_before_tool,
    after_tool_callback=audit_after_tool,
)

r = Runner(app=App(name=APP, root_agent=audited), session_service=InMemorySessionService())
sid = await new_session(r)
print(await ask(r, "查一下訂單 B-2002", session_id=sid))

print("\n--- 稽核記錄 ---")
for line in AUDIT:
    print(" ", line)

訂單 B-2002 的狀態為：已出貨。

--- 稽核記錄 ---
  呼叫 get_order_status args={'order_id': 'B-2002'}
  get_order_status 回傳 {'order_id': 'B-2002', 'status': '已出貨'}


### 短路：把 callback 當護欄用

`before_tool_callback` 回傳一個 dict，工具就**不會真的被執行**，
模型收到的是你給的那個 dict。

In [6]:
BLOCKED_ORDERS = {"X-9999"}


def guard(tool: BaseTool, args: dict, tool_context: ToolContext):
    # 注意工具名稱要跟實際掛上去的那個一致。比錯名字不會報錯，
    # 只會安靜地放行——這是護欄最危險的失敗方式。
    if tool.name == "get_order_status_loud" and args.get("order_id") in BLOCKED_ORDERS:
        return {"error": "這筆訂單受保護，無法查詢。請聯繫主管。"}
    return None


def get_order_status_loud(order_id: str) -> dict:
    """查詢訂單狀態。

    Args:
        order_id: 訂單編號。
    """
    print(f"    ⚠️ 工具真的被執行了：{order_id}")
    return {"order_id": order_id, "status": "已出貨"}


guarded = LlmAgent(
    name="guarded_support",
    model=get_model(),
    instruction="你是客服，查訂單一定要用工具。用繁體中文回答。",
    tools=[get_order_status_loud],
    before_tool_callback=guard,
)
rg = Runner(app=App(name=APP, root_agent=guarded), session_service=InMemorySessionService())

sid = await new_session(rg)
print("正常訂單:", await ask(rg, "查訂單 A-1001", session_id=sid))
print()
sid = await new_session(rg)
print("受保護訂單:", await ask(rg, "查訂單 X-9999", session_id=sid))

    ⚠️ 工具真的被執行了：A-1001


正常訂單: 您的訂單 A-1001 目前的狀態是：「已出貨」。



受保護訂單: 這筆訂單（X-9999）受保護，無法直接查詢。請聯繫主管以獲取進一步協助。


注意受保護那次，`⚠️ 工具真的被執行了` **沒有印出來**——護欄擋在工具之前，
而不是靠 prompt 拜託模型不要查。這是「把政策寫死在程式裡」跟
「寫在 instruction 裡祈禱模型聽話」的差別。

> **⚠️ 護欄最危險的失敗方式**：`guard` 裡比對的 `tool.name` 如果打錯
> （例如工具其實叫 `get_order_status_loud`，你寫成 `get_order_status`），
> 條件永遠是 False，callback 安靜地回傳 `None`，**護欄形同不存在，
> 而且完全不會報錯**。寫護欄一定要有一個「應該被擋下來」的測試案例。

## 3. Plugin：同樣的鉤子，全域生效

Callback 的問題是**它綁在單一 agent 上**。有 20 個 agent 就要掛 20 次。

Plugin 用的是同一套鉤子，但註冊在 `App` 上，整個應用通吃。
而且 Plugin 的鉤子比 agent callback 更多：

In [7]:
from google.adk.plugins.base_plugin import BasePlugin

print("BasePlugin 的鉤子:")
for name in sorted(m for m in dir(BasePlugin) if m.endswith("_callback")):
    print(f"  {name}")

BasePlugin 的鉤子:
  after_agent_callback
  after_model_callback
  after_run_callback
  after_tool_callback
  before_agent_callback
  before_model_callback
  before_run_callback
  before_tool_callback
  on_agent_error_callback
  on_event_callback
  on_model_error_callback
  on_run_error_callback
  on_tool_error_callback
  on_user_message_callback


多出來的 `before_run_callback` / `after_run_callback` /
`on_user_message_callback` / `on_event_callback` 是**整次執行**層級的，
agent callback 看不到這一層。

## 4. ⚠️ 本章最重要的一件事：Plugin 會蓋掉你的 Callback

規則是：

> **Plugin 的 callback 一定先於物件層級的 callback 執行；
> 而且只要 Plugin 回傳了非 `None` 的值，物件的 callback 就完全不會被呼叫。**

這條規則光看文字很容易記反。直接跑一次：

In [8]:
ORDER_LOG = []


class TracerPlugin(BasePlugin):
    """只記錄順序、不干預（全部回 None）。"""

    def __init__(self, label: str):
        super().__init__(name=f"tracer_{label}")
        self.label = label

    async def before_tool_callback(self, *, tool, tool_args, tool_context):
        ORDER_LOG.append(f"PLUGIN[{self.label}].before_tool")
        return None

    async def after_tool_callback(self, *, tool, tool_args, tool_context, result):
        ORDER_LOG.append(f"PLUGIN[{self.label}].after_tool")
        return None


def agent_before_tool(tool: BaseTool, args: dict, tool_context: ToolContext):
    ORDER_LOG.append("AGENT.before_tool")
    return None


def agent_after_tool(tool: BaseTool, args: dict, tool_context: ToolContext, tool_response: dict):
    ORDER_LOG.append("AGENT.after_tool")
    return None


ordered_agent = LlmAgent(
    name="ordered",
    model=get_model(),
    instruction="查訂單一定要用工具。用繁體中文簡短回答。",
    tools=[get_order_status],
    before_tool_callback=agent_before_tool,
    after_tool_callback=agent_after_tool,
)

ro = Runner(
    app=App(name=APP, root_agent=ordered_agent, plugins=[TracerPlugin("A")]),
    session_service=InMemorySessionService(),
)
sid = await new_session(ro)
await ask(ro, "查訂單 C-3003", session_id=sid)

print("實際執行順序：")
for i, line in enumerate(ORDER_LOG, 1):
    print(f"  {i}. {line}")

實際執行順序：
  1. PLUGIN[A].before_tool
  2. AGENT.before_tool
  3. PLUGIN[A].after_tool
  4. AGENT.after_tool


讀一下這個順序，有一點違反直覺：

```
1. PLUGIN.before_tool
2. AGENT.before_tool
     （工具執行）
3. PLUGIN.after_tool     ← 注意這裡
4. AGENT.after_tool
```

一般的「洋蔥式」中介層（像 web middleware）進去的順序是 A→B，
出來會反過來變成 B→A。**ADK 不是這樣**——
不論 before 還是 after，**Plugin 一律排在 agent callback 前面**。

所以如果你想在 agent 的 `after_tool_callback` 裡改寫工具結果，
要知道 Plugin 已經先看過（也可能已經改過）它了。

現在看短路。讓 Plugin 回傳一個值：

In [9]:
ORDER_LOG.clear()


class BlockingPlugin(BasePlugin):
    """回傳非 None → 短路。"""

    def __init__(self):
        super().__init__(name="blocker")

    async def before_tool_callback(self, *, tool, tool_args, tool_context):
        ORDER_LOG.append("PLUGIN.before_tool → 回傳結果（短路！）")
        return {"error": "全域政策：這個工具目前停用中。"}


blocked_runner = Runner(
    app=App(name=APP, root_agent=ordered_agent, plugins=[BlockingPlugin()]),
    session_service=InMemorySessionService(),
)
sid = await new_session(blocked_runner)
answer = await ask(blocked_runner, "查訂單 D-4004", session_id=sid)

print("實際執行順序：")
for i, line in enumerate(ORDER_LOG, 1):
    print(f"  {i}. {line}")
print(f"\n模型的回答：{answer}")

實際執行順序：
  1. PLUGIN.before_tool → 回傳結果（短路！）
  2. AGENT.after_tool

模型的回答：非常抱歉，目前查詢訂單的系統工具暫時停用，無法為您查詢訂單 D-4004 的狀態。請稍後再試或聯繫客服人員協助。


`AGENT.before_tool` **完全沒有出現**（`AGENT.after_tool` 倒是還會跑）。

這就是那條規則的實際後果：

> 你在 agent 上掛的稽核 callback，可能因為某個 plugin 短路而**一次都沒被呼叫**，
> 而且不會有任何錯誤訊息。

**實務結論**：安全護欄、稽核這類「絕對不能被跳過」的邏輯，要寫成 **Plugin**，
不要寫成 agent callback。

## 5. ADK 內建的 Plugin

不用什麼都自己寫，ADK 附了幾個常用的：

In [10]:
import google.adk.plugins as plugins_mod

print("內建 Plugin:")
for name in sorted(n for n in dir(plugins_mod) if n.endswith("Plugin")):
    print(f"  {name}")

內建 Plugin:
  BasePlugin


最實用的是 `ReflectAndRetryToolPlugin`：工具丟出例外時，它會把錯誤訊息
交回給模型，讓模型自己修正參數重試，而不是整個流程直接掛掉。

In [11]:
from google.adk.plugins import ReflectAndRetryToolPlugin

CALLS = []


def strict_lookup(order_id: str) -> dict:
    """查詢訂單。訂單編號必須是 'ORD-' 開頭。

    Args:
        order_id: 訂單編號，格式為 ORD-XXXX。
    """
    CALLS.append(order_id)
    if not order_id.startswith("ORD-"):
        raise ValueError(f"訂單編號格式錯誤：{order_id}，正確格式是 ORD-XXXX")
    return {"order_id": order_id, "status": "處理中"}


retry_agent = LlmAgent(
    name="retry_agent",
    model=get_model(),
    instruction="你是客服。查訂單一定要用 strict_lookup 工具。用繁體中文回答。",
    tools=[strict_lookup],
)

rr = Runner(
    app=App(
        name=APP,
        root_agent=retry_agent,
        plugins=[ReflectAndRetryToolPlugin(max_retries=3)],
    ),
    session_service=InMemorySessionService(),
)
sid = await new_session(rr)
print(await ask(rr, "幫我查編號 5566 的訂單", session_id=sid))
print(f"\n工具實際被呼叫的參數: {CALLS}")

訂單編號 ORD-5566 的狀態目前是「處理中」。

工具實際被呼叫的參數: ['ORD-5566']


如果模型第一次傳了 `5566`（格式錯），plugin 會把錯誤訊息餵回去，
模型看到「正確格式是 ORD-XXXX」就會改成 `ORD-5566` 重試。

沒有這個 plugin 的話，第一次 `ValueError` 就會讓整次執行失敗。

## 6. 順帶一提：`context_cache_config`

第 01 章跑多 agent 時，ADK 可能印過這段提醒：

> *App can transfer between agents but has no `context_cache_config`. Every
> transfer swaps the system instruction and the tool set, so the request prefix
> changes and the whole prompt is re-sent uncached after each transfer.*

每次交棒都會換掉 system instruction 和工具清單，導致 prompt 前綴改變、
快取全部失效。設定 `context_cache_config` 可以讓每個 agent 有自己的快取。

In [12]:
from google.adk.agents.context_cache_config import ContextCacheConfig

cached_app = App(
    name=APP,
    root_agent=agent,
    context_cache_config=ContextCacheConfig(),
)
print("預設快取設定:", cached_app.context_cache_config)

預設快取設定: ContextCacheConfig(cache_intervals=10, ttl=1800s, min_tokens=0, create_http_options=None)


細節（TTL、最小 token 數、怎麼確認快取真的生效）在 30 天實作的 **Day 11**。

## 本章重點

- **`App` 是 ADK 2.x 的應用層容器**。壓縮、快取、可恢復性、plugins
  都設在 `App` 上，設在 agent 上沒有用。
- **Callback 回傳 `None` ＝放行，回傳物件 ＝短路**。護欄就是這樣做的，
  而且擋在工具之前，比在 instruction 裡拜託模型可靠得多。
- **Plugin 用同一套鉤子但全域生效**，而且多了 run 層級的鉤子。
- **⚠️ Plugin 一定先於 agent callback**——**before 和 after 都是**，不是洋蔥式的反序；
  而且 Plugin 短路後，agent 的 `before_*` 完全不會被呼叫。
  絕對不能被跳過的邏輯要寫成 Plugin。
- **`ReflectAndRetryToolPlugin`** 讓模型能從工具錯誤中自我修正。

## 動手練習

1. 把第 4 節的 `BlockingPlugin` 改成回傳 `None`，重跑，
   確認 `2️⃣ AGENT.before_tool` 就回來了。
2. 掛兩個 Plugin，其中第一個短路。第二個 Plugin 的 callback 還會被呼叫嗎？
3. 用 `before_model_callback` 寫一個「輸入含有身分證字號就擋下來」的 Plugin，
   測試它是否比寫在 instruction 裡可靠。

---
**下一站 → `07_workflow_agents.ipynb`**：Layer 3 開始——
Sequential、Parallel、Loop 三種內建的流程編排。